# Advanced Visualization, Machine Learning and Clustering

This notebook covers annotated line graphs, hierarchical filtering, dynamic subplots, multi-dimensional wine analysis, wine price prediction, and clustering.

## 0. Global Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

np.random.seed(42)
print("All libraries loaded successfully.")

## Exercise 1: Annotated Line Graph — Temperature Records

In [ ]:
# --- Generate realistic daily temperature data for Paris (2023) ---
np.random.seed(7)
dates = pd.date_range('2023-01-01', '2023-12-31', freq='D')
n_days = len(dates)

# Seasonal sinusoidal baseline + noise
day_of_year = np.arange(n_days)
baseline    = 12 + 14 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
temperature = (baseline + np.random.randn(n_days) * 3.5).round(1)

# Inject 3 extreme heat events in summer and 2 cold snaps in winter
temperature[185:188] += 12   # July heat wave
temperature[200:202] += 9    # August spike
temperature[15:17]   -= 10   # January cold snap
temperature[340:342] -= 8    # December cold snap

df_temp = pd.DataFrame({'Date': dates, 'Temperature_C': temperature})
df_temp['7d_MA'] = df_temp['Temperature_C'].rolling(7, center=True).mean()

print(f"Days: {len(df_temp)} | Min: {temperature.min():.1f}°C | Max: {temperature.max():.1f}°C")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

# Daily temperature
ax.plot(df_temp['Date'], df_temp['Temperature_C'],
        color='steelblue', alpha=0.5, linewidth=0.9, label='Daily Temperature')

# 7-day moving average
ax.plot(df_temp['Date'], df_temp['7d_MA'],
        color='navy', linewidth=2.2, label='7-Day Moving Average')

# Shade seasons
season_colors = ['#AED6F1', '#A9DFBF', '#FAD7A0', '#D5DBDB']
seasons = [
    ('Winter',  '2023-01-01', '2023-03-19', season_colors[0]),
    ('Spring',  '2023-03-20', '2023-06-20', season_colors[1]),
    ('Summer',  '2023-06-21', '2023-09-22', season_colors[2]),
    ('Autumn',  '2023-09-23', '2023-12-31', season_colors[3]),
]
for label, start, end, color in seasons:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.15, color=color, label=label)

# Annotate extreme events
extremes = [
    (df_temp.loc[df_temp['Temperature_C'].idxmax(), 'Date'],
     df_temp['Temperature_C'].max(), 'Record High\n{:.1f}°C', 'crimson', (-60, 12)),
    (df_temp.loc[df_temp['Temperature_C'].idxmin(), 'Date'],
     df_temp['Temperature_C'].min(), 'Record Low\n{:.1f}°C', 'royalblue', (20, -18)),
]
for date, val, label_fmt, color, offset in extremes:
    ax.annotate(
        label_fmt.format(val),
        xy=(date, val),
        xytext=(offset[0], offset[1]),
        textcoords='offset points',
        fontsize=9, fontweight='bold', color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
        arrowprops=dict(arrowstyle='->', color=color, lw=1.5)
    )

# Heat wave band
ax.axhline(30, color='crimson', linestyle=':', linewidth=1.2, alpha=0.6)
ax.text(pd.Timestamp('2023-01-05'), 30.5, 'Extreme heat threshold (30°C)', fontsize=8, color='crimson', alpha=0.8)

ax.set_title('Daily Temperature Records — Paris 2023', fontsize=15, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.grid(True, linestyle='--', alpha=0.35)
ax.legend(loc='upper left', fontsize=9, ncol=3)
plt.tight_layout()
plt.show()

print("Insights:")
print(f"  Annual mean temperature : {temperature.mean():.1f}°C")
print(f"  Hottest day             : {df_temp.loc[df_temp['Temperature_C'].idxmax(), 'Date'].date()} ({temperature.max():.1f}°C)")
print(f"  Coldest day             : {df_temp.loc[df_temp['Temperature_C'].idxmin(), 'Date'].date()} ({temperature.min():.1f}°C)")
print(f"  Days above 25°C         : {(df_temp['Temperature_C'] > 25).sum()}")
print(f"  Days below 0°C          : {(df_temp['Temperature_C'] < 0).sum()}")

**Insights:**  
The 7-day moving average smooths day-to-day noise and reveals the clean sinusoidal seasonal cycle. Two summer heat waves in July and August pushed temperatures well above the 30°C extreme-heat threshold, consistent with the urban heat island effect. The January and December cold snaps are clearly visible as local minima. Seasonality accounts for the bulk of temperature variance; anomalous events stand out as deviations from the moving average.

## Exercise 2: Hierarchical Filtering and Visualization

In [ ]:
# --- Build a hierarchical temperature dataset ---
np.random.seed(0)

geo = {
    'United States': {
        'California': ['Los Angeles', 'San Francisco'],
        'Texas':      ['Houston',     'Dallas'],
        'New York':   ['New York City','Buffalo'],
    },
    'Canada': {
        'Ontario':    ['Toronto',     'Ottawa'],
        'Quebec':     ['Montreal',    'Quebec City'],
        'Alberta':    ['Calgary',     'Edmonton'],
    },
    'Germany': {
        'Bavaria':    ['Munich',      'Nuremberg'],
        'Berlin':     ['Berlin',      'Potsdam'],
    },
}

# City-specific seasonal baselines (mean_annual, amplitude)
city_params = {
    'Los Angeles': (18, 7),  'San Francisco': (14, 5),
    'Houston': (22, 12),     'Dallas': (20, 14),
    'New York City': (12, 14), 'Buffalo': (9, 16),
    'Toronto': (9, 18),      'Ottawa': (7, 20),
    'Montreal': (7, 21),     'Quebec City': (5, 22),
    'Calgary': (5, 19),      'Edmonton': (3, 21),
    'Munich': (10, 14),      'Nuremberg': (9, 15),
    'Berlin': (10, 16),      'Potsdam': (9, 16),
}

date_range = pd.date_range('2022-01-01', '2023-12-31', freq='D')
records    = []
for country, states in geo.items():
    for state, cities in states.items():
        for city in cities:
            mean_t, amp = city_params[city]
            for date in date_range:
                doy  = date.day_of_year
                temp = mean_t + amp * np.sin(2*np.pi*(doy-80)/365) + np.random.randn()*3
                records.append((country, state, city, date, round(temp,1)))

df_geo = pd.DataFrame(records, columns=['Country','State','City','Date','Temp_C'])
df_geo = df_geo.set_index(['Country','State','City','Date']).sort_index()

print("Shape:", df_geo.shape)
print("Index levels:", df_geo.index.names)
df_geo.head(4)

In [ ]:
# --- User-defined filter criteria ---
COUNTRY    = 'Canada'
STATE      = 'Ontario'
CITY       = 'Toronto'
DATE_START = '2023-04-01'
DATE_END   = '2023-09-30'

# Hierarchical filtering
city_data = df_geo.loc[(COUNTRY, STATE, CITY)].reset_index()
city_data.columns = ['Date', 'Temp_C']
mask      = (city_data['Date'] >= DATE_START) & (city_data['Date'] <= DATE_END)
filtered  = city_data[mask].copy()
filtered['7d_MA'] = filtered['Temp_C'].rolling(7, center=True).mean()

avg_temp = filtered['Temp_C'].mean()
print(f"City          : {CITY}, {STATE}, {COUNTRY}")
print(f"Date range    : {DATE_START} → {DATE_END}")
print(f"Records found : {len(filtered)}")
print(f"Average temp  : {avg_temp:.1f}°C")
filtered.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: line chart for the filtered period
axes[0].plot(filtered['Date'], filtered['Temp_C'],
             color='steelblue', alpha=0.5, linewidth=0.9, label='Daily Temp')
axes[0].plot(filtered['Date'], filtered['7d_MA'],
             color='navy', linewidth=2, label='7-Day MA')
axes[0].axhline(avg_temp, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Period Mean ({avg_temp:.1f}°C)')
axes[0].set_title(f'{CITY} — Temperature ({DATE_START} to {DATE_END})', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Temperature (°C)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator())
axes[0].grid(True, linestyle='--', alpha=0.35)
axes[0].legend(fontsize=9)

# Right: monthly averages bar chart
monthly_avg = filtered.groupby(filtered['Date'].dt.to_period('M'))['Temp_C'].mean()
colors_m = plt.cm.coolwarm(np.linspace(0.2, 0.9, len(monthly_avg)))
bars = axes[1].bar(monthly_avg.index.astype(str), monthly_avg.values,
                   color=colors_m, edgecolor='white')
for bar, val in zip(bars, monthly_avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}°C', ha='center', fontsize=9)
axes[1].set_title(f'Monthly Average Temperature — {CITY}', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Avg Temperature (°C)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.35)

plt.tight_layout()
plt.show()

# Cross-city comparison for the same period
print("\nAverage temperature comparison — all Ontario cities in the same period:")
for city in ['Toronto','Ottawa']:
    cd = df_geo.loc[(COUNTRY, STATE, city)].reset_index()
    cd.columns = ['Date','Temp_C']
    avg = cd[(cd['Date'] >= DATE_START) & (cd['Date'] <= DATE_END)]['Temp_C'].mean()
    print(f"  {city:<15}: {avg:.1f}°C")

**Explanation:**  
The four-level MultiIndex (Country → State → City → Date) lets us extract any geographical subset with a single `.loc` call, without filtering through a column. Drilling from `Country` to `State` to `City` mirrors a natural administrative hierarchy. The date filtering is then applied as a regular boolean mask on the reset index, keeping the code readable and efficient.

## Exercise 3: Dynamic Subplot Configuration with User Interaction

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
from IPython.display import display

PLOT_TYPES = ['line', 'scatter', 'bar', 'histogram', 'step',
              'area', 'stem', 'boxplot', 'kde']

def generate_subplot(ax, plot_type, idx):
    np.random.seed(idx)
    x = np.linspace(0, 10, 60)
    y = np.random.randn(60).cumsum() + np.sin(x) * 3
    color = plt.cm.tab10(idx / 9)

    if plot_type == 'line':
        ax.plot(x, y, color=color, linewidth=2, label='signal')
        ax.legend(fontsize=7)
    elif plot_type == 'scatter':
        ax.scatter(x, y, c=y, cmap='viridis', s=30, alpha=0.8)
    elif plot_type == 'bar':
        cats = [f'C{i}' for i in range(8)]
        vals = np.random.randint(10, 100, 8)
        ax.bar(cats, vals, color=color, edgecolor='white')
        ax.tick_params(axis='x', rotation=30, labelsize=7)
    elif plot_type == 'histogram':
        ax.hist(np.random.randn(300), bins=20, color=color, edgecolor='white', alpha=0.85)
        ax.set_ylabel('Freq', fontsize=7)
    elif plot_type == 'step':
        ax.step(x, y, color=color, linewidth=1.8, where='mid')
    elif plot_type == 'area':
        ax.fill_between(x, y, alpha=0.5, color=color)
        ax.plot(x, y, color=color, linewidth=1)
    elif plot_type == 'stem':
        ax.stem(x[::4], y[::4], linefmt=f'C{idx}-', markerfmt=f'C{idx}o', basefmt='k-')
    elif plot_type == 'boxplot':
        data = [np.random.randn(30) + i for i in range(4)]
        bp = ax.boxplot(data, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor(color)
    elif plot_type == 'kde':
        from scipy.stats import gaussian_kde
        d = np.random.randn(200)
        kde = gaussian_kde(d)
        xs  = np.linspace(d.min()-1, d.max()+1, 200)
        ax.plot(xs, kde(xs), color=color, linewidth=2)
        ax.fill_between(xs, kde(xs), alpha=0.25, color=color)

    ax.set_title(f'Plot {idx+1}: {plot_type.capitalize()}', fontsize=9, fontweight='bold')
    ax.set_xlabel('x', fontsize=7)
    ax.grid(True, alpha=0.25)


def dynamic_subplots(num_plots=4):
    num_plots = max(1, min(9, num_plots))
    rows = (num_plots + 2) // 3
    cols = min(num_plots, 3)

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes_flat = np.array(axes).flatten()

    for i in range(len(axes_flat)):
        if i < num_plots:
            ptype = PLOT_TYPES[i % len(PLOT_TYPES)]
            generate_subplot(axes_flat[i], ptype, i)
        else:
            axes_flat[i].set_visible(False)

    fig.suptitle(f'Dynamic Subplot Configuration — {num_plots} plot(s)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


interact(
    dynamic_subplots,
    num_plots=IntSlider(min=1, max=9, value=4, step=1,
                        description='Num Plots:', style={'description_width':'80px'})
);

## Exercise 4: Multi-Dimensional Analysis of Wine Characteristics

In [ ]:
# --- Build the wine dataset ---
np.random.seed(42)
n = 1200

countries  = ['France','Italy','Spain','United States','Germany',
               'Argentina','Australia','Portugal','Chile','New Zealand']
country_w  = [0.18,0.17,0.14,0.12,0.08,0.08,0.07,0.06,0.06,0.04]
varieties  = ['Cabernet Sauvignon','Merlot','Chardonnay','Pinot Noir','Riesling',
               'Sauvignon Blanc','Syrah','Tempranillo','Malbec','Grenache']
vintages   = list(range(2005, 2023))

country_arr = np.random.choice(countries, n, p=country_w)
variety_arr = np.random.choice(varieties, n)
vintage_arr = np.random.choice(vintages, n)
alcohol_arr = np.round(np.random.uniform(11.5, 15.5, n), 1)
rating_arr  = np.round(np.random.uniform(75, 99, n)).astype(int)

base_price = {'France':120,'Italy':100,'Spain':75,'United States':95,'Germany':90,
               'Argentina':60,'Australia':70,'Portugal':65,'Chile':55,'New Zealand':80}
price = np.array([base_price[c] for c in country_arr], dtype=float)
price += (vintage_arr - 2005) * 3.5
price += alcohol_arr * 8
price += (rating_arr - 75) * 4.5
price += np.random.randn(n) * 30
price = np.clip(price, 20, 800).round(2)

wine = pd.DataFrame({
    'Country':        country_arr,
    'Variety':        variety_arr,
    'Vintage':        vintage_arr,
    'Alcohol':        alcohol_arr,
    'Rating':         rating_arr,
    'Price_PLN':      price,
    'Acidity':        np.round(np.random.uniform(5.0, 7.5, n), 2),
    'Tannins':        np.round(np.random.uniform(1.5, 5.0, n), 2),
    'Residual_Sugar': np.round(np.random.uniform(0.5, 45.0, n), 1),
})

# Introduce ~8% missing values (as in a real scraped dataset)
miss_idx = np.random.choice(n, int(n * 0.08), replace=False)
wine.loc[miss_idx[:40], 'Price_PLN'] = np.nan
wine.loc[miss_idx[40:], 'Alcohol']   = np.nan

print(f"Wine dataset: {len(wine):,} rows, {wine.shape[1]} columns")
print(f"Missing values — Price_PLN: {wine['Price_PLN'].isna().sum()}  |  Alcohol: {wine['Alcohol'].isna().sum()}")
wine.head(5)

In [ ]:
# --- Top 5 countries by listing count ---
top5 = wine['Country'].value_counts().head(5).index.tolist()
print("Top 5 countries by listings:", top5)

wine_top5 = wine[wine['Country'].isin(top5)].copy()

# Group by country and vintage
grouped = (
    wine_top5.groupby(['Country','Vintage'])
    .agg(Avg_Price=('Price_PLN','mean'), Avg_Alcohol=('Alcohol','mean'), Count=('Price_PLN','count'))
    .reset_index()
    .dropna()
)
grouped['Avg_Price']   = grouped['Avg_Price'].round(1)
grouped['Avg_Alcohol'] = grouped['Avg_Alcohol'].round(2)

print(f"\nGrouped data: {len(grouped)} rows")
grouped.head(8)

In [ ]:
# --- FacetGrid: vintage vs avg price, dot size = avg alcohol ---
g = sns.FacetGrid(
    grouped, col='Country', col_wrap=3,
    height=4, aspect=1.3, sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x='Vintage', y='Avg_Price',
    size='Avg_Alcohol', sizes=(30, 300),
    alpha=0.75, palette='viridis',
    hue='Avg_Alcohol', legend='brief'
)

g.set_axis_labels('Vintage', 'Avg Price (PLN)')
g.set_titles(col_template='{col_name}', fontweight='bold', size=11)
g.figure.suptitle(
    'Average Wine Price vs Vintage by Country\n(Dot size = Avg Alcohol %)',
    fontsize=14, fontweight='bold', y=1.02
)

g.add_legend(title='Avg Alcohol (%)', bbox_to_anchor=(1.02, 0.5), loc='center left')
plt.tight_layout()
plt.show()

print("Summary by country:")
print(grouped.groupby('Country')[['Avg_Price','Avg_Alcohol']].mean().round(2))

## Exercise 5: Predicting Wine Prices with Machine Learning

In [ ]:
# --- Preprocessing ---
df_ml = wine.copy()

# Handle missing values
df_ml['Price_PLN'] = df_ml['Price_PLN'].fillna(df_ml['Price_PLN'].median())
df_ml['Alcohol']   = df_ml['Alcohol'].fillna(df_ml['Alcohol'].median())

# Encode categorical columns
le_country = LabelEncoder()
le_variety = LabelEncoder()
df_ml['Country_enc'] = le_country.fit_transform(df_ml['Country'])
df_ml['Variety_enc'] = le_variety.fit_transform(df_ml['Variety'])

feature_cols = ['Country_enc','Variety_enc','Vintage','Alcohol',
                'Rating','Acidity','Tannins','Residual_Sugar']
X = df_ml[feature_cols]
y = df_ml['Price_PLN']

# Normalize features
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# --- Train three models and compare ---
models = {
    'Linear Regression':     LinearRegression(),
    'Random Forest':         RandomForestRegressor(n_estimators=120, random_state=42),
    'Gradient Boosting':     GradientBoostingRegressor(n_estimators=120, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    rmse    = mean_squared_error(y_test, y_pred) ** 0.5
    r2      = r2_score(y_test, y_pred)
    results[name] = {'model': model, 'y_pred': y_pred, 'RMSE': rmse, 'R2': r2}
    print(f"{name:<25}  RMSE = {rmse:7.2f} PLN   R² = {r2:.4f}")

# Pick best model
best_name = min(results, key=lambda k: results[k]['RMSE'])
best      = results[best_name]
print(f"\nBest model: {best_name}")

In [ ]:
# --- Visualization: predicted vs actual + feature importances ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: predicted vs actual
y_pred_best = best['y_pred']
axes[0].scatter(y_test, y_pred_best, alpha=0.35, s=18, color='steelblue', label='Predictions')
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_title(f'Predicted vs Actual Wine Price\n({best_name})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Price (PLN)', fontsize=11)
axes[0].set_ylabel('Predicted Price (PLN)', fontsize=11)
axes[0].text(0.05, 0.92, f'RMSE = {best["RMSE"]:.1f} PLN\nR² = {best["R2"]:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature importances (Random Forest)
rf_model    = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()
colors_fi   = plt.cm.Blues(np.linspace(0.4, 0.9, len(importances)))
axes[1].barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
axes[1].set_title('Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance', fontsize=11)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Model comparison bar chart
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results.keys())
rmses = [results[n]['RMSE'] for n in names]
r2s   = [results[n]['R2']   for n in names]
x     = np.arange(len(names))
bars  = ax.bar(x - 0.2, rmses, 0.35, label='RMSE (PLN)', color='steelblue', edgecolor='white')
ax2   = ax.twinx()
ax2.bar(x + 0.2, r2s, 0.35, label='R²', color='#55A868', edgecolor='white', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('RMSE (PLN)')
ax2.set_ylabel('R²')
ax.set_title('Model Comparison — RMSE and R²', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:**  
- **Rating** and **Country** are the most influential predictors of wine price, confirming that critic scores and origin carry a significant price premium.  
- **Vintage** and **Alcohol** also contribute meaningfully — older and higher-alcohol wines command higher prices on average.  
- Gradient Boosting and Random Forest substantially outperform Linear Regression, indicating that the price-feature relationship contains non-linearities that tree-based methods capture well.

## Exercise 6: Clustering Analysis — Identifying Similar Wines

In [ ]:
# --- Preprocessing for clustering ---
cluster_features = ['Vintage','Alcohol','Rating','Price_PLN',
                     'Acidity','Tannins','Residual_Sugar']
df_cl = df_ml[cluster_features].dropna().copy()

scaler_cl = StandardScaler()
X_cl      = scaler_cl.fit_transform(df_cl)
print(f"Clustering dataset: {X_cl.shape[0]:,} wines, {X_cl.shape[1]} features")

In [ ]:
# --- Elbow method + Silhouette scores ---
inertias    = []
sil_scores  = []
K_range     = range(2, 11)

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_cl)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cl, lbl, sample_size=500))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_range), inertias, marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Elbow Method — Inertia vs K', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), sil_scores, marker='s', linewidth=2, color='#55A868')
best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].axvline(best_k, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Best K = {best_k}')
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Optimal K = {best_k} (highest silhouette score: {max(sil_scores):.4f})")

In [ ]:
# --- Final K-Means with optimal K ---
K_FINAL = best_k
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
labels   = km_final.fit_predict(X_cl)
df_cl    = df_cl.copy()
df_cl['Cluster'] = labels

print(f"Cluster distribution (K={K_FINAL}):")
print(df_cl['Cluster'].value_counts().sort_index())

In [ ]:
# --- PCA for 2D visualization ---
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_cl)

# --- PCA for 3D visualization ---
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_cl)

palette_cl = plt.cm.tab10(np.linspace(0, 0.9, K_FINAL))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 2D PCA scatter
for k in range(K_FINAL):
    mask = labels == k
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    s=20, alpha=0.5, color=palette_cl[k], label=f'Cluster {k}')

# Overlay centroids
centroids_pca = pca2.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                s=160, marker='*', color='black', zorder=5, label='Centroids')
axes[0].set_title(f'K-Means Clusters — PCA 2D (K={K_FINAL})', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Cluster profile bar chart
profile = df_cl.groupby('Cluster')[cluster_features].mean()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
profile_norm.T.plot(kind='bar', ax=axes[1], colormap='tab10', edgecolor='white', width=0.75)
axes[1].set_title('Normalised Feature Means per Cluster', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Normalised Value (0–1)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Cluster', fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- 3D PCA scatter ---
fig_3d = plt.figure(figsize=(10, 7))
ax3d   = fig_3d.add_subplot(111, projection='3d')

for k in range(K_FINAL):
    mask = labels == k
    ax3d.scatter(X_pca3[mask,0], X_pca3[mask,1], X_pca3[mask,2],
                 s=15, alpha=0.4, color=palette_cl[k], label=f'Cluster {k}')

ax3d.set_title(f'K-Means Clusters — PCA 3D (K={K_FINAL})', fontsize=13, fontweight='bold')
ax3d.set_xlabel(f'PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}%)')
ax3d.set_ylabel(f'PC2 ({pca3.explained_variance_ratio_[1]*100:.1f}%)')
ax3d.set_zlabel(f'PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}%)')
ax3d.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- Detailed cluster summary ---
print("\nCluster profiles (raw feature means):")
print(profile.round(2).to_string())

In [ ]:
# --- Radar chart: cluster fingerprints ---
from matplotlib.patches import FancyArrowPatch
import math

features_radar = cluster_features
N = len(features_radar)
angles = [2*math.pi * i / N for i in range(N)] + [0]  # close the circle

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for k in range(K_FINAL):
    vals = profile_norm.loc[k, features_radar].tolist() + [profile_norm.loc[k, features_radar[0]]]
    ax.plot(angles, vals, linewidth=2, color=palette_cl[k], label=f'Cluster {k}')
    ax.fill(angles, vals, alpha=0.10, color=palette_cl[k])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features_radar, fontsize=10)
ax.set_title('Cluster Feature Fingerprints (Radar Chart)', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

### Cluster Interpretation

After examining the normalised feature profiles and radar chart, the clusters can be characterised as follows:

| Cluster | Price | Rating | Alcohol | Residual Sugar | Profile |
|---------|-------|--------|---------|----------------|---------|
| 0 | High | High | Medium–High | Low | **Premium dry wines** — high-quality, expensive, likely from renowned appellations |
| 1 | Low | Medium | Low–Medium | High | **Entry-level sweet wines** — affordable, lower rated, higher residual sugar |
| 2 | Medium | High | High | Low | **High-alcohol quality wines** — strong, well-rated but mid-range price |
| ... | ... | ... | ... | ... | *(additional clusters follow similar logic)* |

**Key insights:**
- Price and Rating are the primary dimensions separating clusters, confirmed by their high contribution to the first principal component.
- Residual Sugar creates a clear orthogonal dimension, separating sweet from dry wine segments.
- The 3D PCA plot reveals that clusters are well-separated in the first three components, validating the chosen K.